# Carousel — a quantitative teardown 🔬
### Sector-momentum spread · rotation alpha over the equal-weight basket (HAC) · the long-short factor · the top-k sweep

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Beats the basket?: Not supported](https://img.shields.io/badge/Beats_the_basket%3F-Not_supported-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim with its standard error.* The steelman is §4.1 sector momentum rotation. We prove the engine on a synthetic sector panel with baked momentum, then show the real SPDR sectors carry no rotation edge over the basket.

> ⚠️ **Not investment advice.** The core executes on synthetic data; the real run is in [`../docs/results.md`](../docs/results.md), sources in [`../docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back to intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from carousel import data, rotation, strategy, decompose, extension

# Offline synthetic sectors: a MOMENTUM panel (leaders persist) and a no-momentum NULL. The real SPDR
# verdict is in ../docs/results.md.
panel,  truth = data.synthetic_sectors(mom_strength=0.0011, seed=28)   # the momentum panel
panel0, _     = data.synthetic_sectors(mom_strength=0.0,    seed=28)   # the null
print(f"{truth.n_sectors} synthetic sectors x {truth.n_bars} days | baked mom_strength={truth.mom_strength} | null=0")


11 synthetic sectors x 4032 days | baked mom_strength=0.0011 | null=0


## Beat 0 · Verdict

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — does the hot sector stay hot? | 🟡 `WEAK` | Strong on the control (alpha HAC *t* ≈ 7.6); on the 11 real SPDR sectors the long-short momentum factor is **-1.5%/yr** (*t* **-0.5**) — no premium. |
| **Tradability** | 🔴 `MIRAGE` | Rotation Sharpe **+0.43** ties the equal-weight basket (**+0.45**, gain **-0.02**) at **7.3×/yr** turnover and extra concentration. |
| **Beats the basket?** | ⚪ `Not supported` | Beats the basket on only **17%** of top-k choices; alpha over the basket *t* = **+0.3**. |

> **In one sentence:** sector momentum is real and recoverable on a synthetic control, but the 11-sector SPDR cross-section carries no momentum premium and rotation merely re-buys the market with turnover and concentration.

*(This notebook executes on the synthetic control; the real SPDR numbers are in [`../docs/results.md`](../docs/results.md).)*

## Beat 1 · The claim, precisely

Momentum score $m_i = \prod_{t-126}^{t-21}(1+r_i)-1$ per sector; the long-only book holds the top-$k$, the long-short is top-minus-bottom-$k$, monthly. The test is the rotation alpha against the **equal-weight basket** $\bar r_t = \frac1N\sum_i r_{i,t}$ — concentration is only worth it if $\alpha>0$. The synthetic bakes a persistent per-sector drift; $\text{mom\_strength}=0$ is the null.

In [2]:
rs = rotation.rotation_strength(panel)
print(f"top-minus-bottom sector spread {rs['top_minus_bottom_ann_pct']:+.1f}%/yr (synthetic); "
      f"null {rotation.rotation_strength(panel0)['top_minus_bottom_ann_pct']:+.1f}%/yr")

top-minus-bottom sector spread +20.5%/yr (synthetic); null +0.5%/yr


## Beat 2 · So what?

Sector rotation trivially *looks* active and clever, but a long-only book of a few sectors has a beta near 1 to the basket, so most of its return is just the market. The decisive tests are therefore relative: the alpha *over* the equal-weight basket, the sign of the long-short factor, and whether any of it survives out of the cherry-picked concentration. Beats 4–6 run all three.

## Beat 3 · Pre-registered protocol

1. **Spread** (`rotation.rotation_strength`): top-minus-bottom forward return.
2. **Alpha over basket** (`decompose.vs_equal_weight`): HAC *t*.
3. **Long-short factor** (`decompose.long_short_tstat`): the pure momentum premium.
4. **Generalisation** (`extension.topk_sweep`); **null** collapses.

**Mirage line:** alpha over basket insignificant, long-short flat, and a coin-flip top-k win-rate.

## Beat 4 · The teardown

### 4a · Rotation vs basket, momentum vs null

In [3]:
for label, p in [('momentum', panel), ('null', panel0)]:
    a = decompose.vs_equal_weight(p, cost_bps=3.0); ls = decompose.long_short_tstat(p, cost_bps=3.0)
    print(f"{label:9s}: alpha over basket {a['alpha_ann_pct']:+.1f}%/yr (HAC t {a['alpha_t']:+.1f}), beta {a['beta']:.2f}; "
          f"long-short {ls['mean_ann_pct']:+.1f}%/yr (t {ls['t_stat']:+.1f})")

momentum : alpha over basket +10.1%/yr (HAC t +7.6), beta 1.00; long-short +19.6%/yr (t +8.9)


null     : alpha over basket -0.7%/yr (HAC t -0.6), beta 1.03; long-short -0.1%/yr (t -0.0)


### 4b · The top-k sweep — robustness or cherry-pick

In [4]:
sw = extension.topk_sweep(panel, cost_bps=3.0)
display(sw['matrix'].round(3))
print(f"synthetic: {sw['frac_beat_ew']:.0%} of top-k beat the basket, mean gain {sw['mean_gain']:+.2f}")
print('Real SPDR sectors: only 17% of top-k beat the basket -- a coin flip (best k=1).')

,rotation_sharpe,ew_sharpe,gain_over_ew
top_k,,,
1,1.3690,0.6400,0.7290
2,1.2720,0.6400,0.6320
3,1.2610,0.6400,0.6210
4,1.1330,0.6400,0.4940
5,1.1720,0.6400,0.5320
6,1.0470,0.6400,0.4070


synthetic: 100% of top-k beat the basket, mean gain +0.57
Real SPDR sectors: only 17% of top-k beat the basket -- a coin flip (best k=1).


> 💡 **In plain words.** If rotation were a real edge it would help whether you held the top 2 sectors or the top 5. On the real sectors it helps at one number and hurts at the others — the signature of fitting the past, not finding a signal.

## Beat 5 · The verdict

- **No premium** (4a): real long-short -1.5%/yr (*t* -0.5).
- **Ties the basket** (4a): alpha over basket *t* +0.3.
- **Cherry-picked** (4b): 17% of top-k beat the basket.

> **Signal `WEAK` · Tradability `MIRAGE` · Beats the basket? `Not supported`.**

## Beat 6 · Could you trade it?

- **Ties do-nothing** before taxes/7.3×-turnover.
- **Paid in concentration risk** for no return.
- **The win is hindsight** — fragile to the number of sectors held.

Tradability **`MIRAGE`**; beats-the-basket **`Not supported`**.

## Beat 7 · Going further

### 7a · Worked complement — the top-k sweep
Rotation's gain over the equal-weight basket across the number of sectors held.

In [5]:
sw = extension.topk_sweep(panel, cost_bps=3.0)
print(f"synthetic momentum: {sw['frac_beat_ew']:.0%} of top-k beat the basket (best k={sw['best_k']}).")
print('Real SPDR sectors (../docs/extension.md): 17% beat the basket -- a rule that helps')
print('at one concentration and not the others is a coin flip with extra turnover, not an edge.')

synthetic momentum: 100% of top-k beat the basket (best k=1).
Real SPDR sectors (../docs/extension.md): 17% beat the basket -- a rule that helps
at one concentration and not the others is a coin flip with extra turnover, not an edge.


**The result.** On the real sectors the rotation-minus-basket gain swings around zero as you vary how many sectors you hold, beating the basket only **17%** of the time. Pick the number after seeing the result and you can always tell a sector-rotation success story; demand it work *out of sample, at an unchosen concentration*, and it's a coin flip. That cherry-pick is exactly what the `Not supported` stamp records. Full run in [`../docs/extension.md`](../docs/extension.md).

### 7b · Other forks
- **A real cross-section** — 49 Fama-French industries instead of 11 SPDRs; sector momentum may need genuine breadth to exist.
- **Dual momentum** — gate rotation on an absolute-momentum (vs cash) filter; does the trend filter help or just add parameters?
- **Risk-parity sectors** vs equal-weight as the benchmark — a fairer bar.

PRs welcome — give rotation breadth, or find the variant that beats buy-the-basket out of sample.